In [ ]:
import os
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#### Functions

In [ ]:
# plot target drift
def plot_target_drift(df_train, df_valid, df_test, str_target, str_filename='output/target_drift.png'):
    list_labels = ['Train', 'Validation', 'Test']
    list_means = [df_train[str_target].mean(), df_valid[str_target].mean(), df_test[str_target].mean()]
    # plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(list_labels, list_means, color='steelblue', edgecolor='black')
    ax.set_title('Target Mean by Split (Target Drift)', fontsize=16)
    ax.set_xlabel('Split', fontsize=12)
    ax.set_ylabel('Target Mean', fontsize=12)
    # annotate bars
    for i, flt_mean in enumerate(list_means):
        ax.text(i, flt_mean + (ax.get_ylim()[1] * 0.01), f'{flt_mean:.4f}', ha='center', fontsize=11)
    # pad the top so labels don't clip
    ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

#### Constants

In [ ]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# s3 path
str_s3_path = f's3://{str_bucket}/00_data_collection/data.csv'
print(f'S3 Path: {str_s3_path}')

# target
str_target = 'default_12m'
print(f'Target: {str_target}')

# date column
str_date_col = 'origination_date'
print(f'Date Column: {str_date_col}')

# split proportions
flt_train = 0.70
flt_valid = 0.15
flt_test = 0.15

# output directory
os.makedirs('output', exist_ok=True)

#### Read Data

In [ ]:
# read data
df = pd.read_csv(str_s3_path)

# convert date column
df[str_date_col] = pd.to_datetime(df[str_date_col])

# sort by date
df = df.sort_values(str_date_col).reset_index(drop=True)

# shape
print(f'Shape: {df.shape}')
df.head()

#### Split Data

In [ ]:
# split indices
int_n_rows = len(df)
int_train_end = int(int_n_rows * flt_train)
int_valid_end = int(int_n_rows * (flt_train + flt_valid))

# split
df_train = df.iloc[:int_train_end].reset_index(drop=True)
df_valid = df.iloc[int_train_end:int_valid_end].reset_index(drop=True)
df_test = df.iloc[int_valid_end:].reset_index(drop=True)

# print shape and date range for each
for str_name, df_split in [('Train', df_train), ('Validation', df_valid), ('Test', df_test)]:
    print(f'{str_name}: {df_split.shape} | {df_split[str_date_col].min().date()} to {df_split[str_date_col].max().date()}')

#### Target Drift

In [ ]:
plot_target_drift(df_train, df_valid, df_test, str_target)

#### Save

In [ ]:
# save to s3 as parquet
for str_name, df_split in [('df_train', df_train), ('df_valid', df_valid), ('df_test', df_test)]:
    str_s3_output = f's3://{str_bucket}/{str_step}/{str_name}.parquet'
    df_split.to_parquet(str_s3_output, index=False)
    print(f'{str_name} saved to {str_s3_output}')